##PVproject1.2 추가기능 (2026/04/06)
* 균일한 일사량 감소 이벤트를 추가했습니다.
* 일교차를 구현했습니다.

##bug수정내용
* 구름 이벤트에서 duration이 끝날 시 구름이 즉시 사라지는 비현실적인 현상을 방지했습니다.
* P&O 실행점이 새로운 GMPP로 이동하지 않는 현상 수정
* 기타 변수 할당 순서 문제 수정

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pvlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 65.2 MB/s eta 0:00:00


In [12]:
import numpy as np
import pandas as pd
import pvlib
from pvlib.location import Location
from tqdm import tqdm
import os
np.random.seed(1045)

# ==========================================
# [Step 1] 기본 시스템 및 위치 설정
# ==========================================
LATITUDE = 37.5665
LONGITUDE = 126.9780
TZ = 'Asia/Seoul'
site_location = Location(LATITUDE, LONGITUDE, tz=TZ, name='Seoul')

START_HOUR = 5
END_HOUR = 19
NUM_MODULES = 10
WINDOW_SIZE = 60
NOCT_INSTALLED = 43.0

cec_modules = pvlib.pvsystem.retrieve_sam('cecmod')
MODULE_DATA = cec_modules['Hanwha_Q_Cells_Q_PEAK_DUO_L_G5_2_390']

# ==========================================
# [Step 2] 구름 이벤트 클래스 (최종 수정본)
# ==========================================
class CloudEvent:
    def __init__(self):
        self.active = False
        self.start_t = 0
        self.duration = 0
        self.intensity = 0
        self.velocity = 0.0
        self.width = 0.0
        self.event_type = 'moving'

    def trigger(self, t):
        self.active = True
        self.start_t = t
        self.intensity = np.random.uniform(0.3, 0.8)

        if np.random.rand() < 0.5:
            self.event_type = 'moving'
            self.velocity = np.random.uniform(0.01, 0.1)
            self.width = np.random.uniform(1.0, 4.0)
            # moving의 경우 duration을 사용하지 않고 '위치'로 종료 판단
        else:
            self.event_type = 'uniform'
            self.duration = np.random.randint(300, 1200) # uniform일 때만 duration 사용
            self.velocity = 0.0
            self.width = 0.0

    def get_shading_factors(self, t):
        if not self.active: return np.ones(NUM_MODULES)

        elapsed = t - self.start_t
        factors = np.ones(NUM_MODULES)

        if self.event_type == 'moving':
            # 1. 이동하는 구름
            # 구름의 중심 위치 계산 (시작점: 0번 패널 앞쪽(-width)에서 출발)
            center = (elapsed * self.velocity) - self.width

            # [핵심 수정] 구름의 꼬리까지 마지막 패널(NUM_MODULES)을 완전히 빠져나갔는지 확인 (2026/04/06)
            if center > (NUM_MODULES + self.width):
                self.active = False
                return np.ones(NUM_MODULES)

            # 구름이 아직 지나가는 중이라면 음영 계산
            for i in range(NUM_MODULES):
                dist = abs(i - center)
                if dist < self.width: # width 범위 내에 들어온 패널만 음영 적용
                    shading = self.intensity * np.exp(-(dist**2) / (2 * (self.width/2)**2))
                    factors[i] = 1 - shading

        elif self.event_type == 'uniform':
            # 2. 전체 패널 균일 흐려짐 (2026/04/06)
            # 설정된 duration(사인파 1주기)이 끝나면 종료
            if elapsed >= self.duration:
                self.active = False
                return np.ones(NUM_MODULES)

            shading = self.intensity * np.sin(np.pi * (elapsed / self.duration))
            factors *= (1 - shading)

        return factors

# ==========================================
# [Step 3] 일 단위 시뮬레이션 함수 (환경 생성 + MPPT 구동)
# ==========================================
def simulate_single_day(target_date):
    print(f"\n[{target_date}] 시뮬레이션 시작...")

    # 1. 해당 날짜의 시간 인덱스 생성
    start_time = f'{target_date} {START_HOUR:02d}:00:00'
    end_time = f'{target_date} {END_HOUR:02d}:00:00'
    times = pd.date_range(start=start_time, end=end_time, freq='s', tz=TZ)
    total_seconds = len(times)

    # 2. 맑은 날 일사량(Clear-sky) 추출
    clearsky = site_location.get_clearsky(times)
    base_g_array = clearsky['ghi'].values

    # 3. 환경 매트릭스 생성 (구름 적용) - 수정된 부분
    G_matrix = np.zeros((total_seconds, NUM_MODULES))
    cloud = CloudEvent()

    # [수정] True/False 대신 정수로 구름 상태를 기록합니다 (0: 맑음, 1: moving, 2: uniform) (2026/04/06)
    cloud_type_array = np.zeros(total_seconds, dtype=int)

    for t in range(total_seconds):
        base_g = max(base_g_array[t], 20.0)
        # 구름 발생
        if not cloud.active and np.random.rand() < (15 / total_seconds):
            cloud.trigger(t)

        factors = cloud.get_shading_factors(t)
        G_matrix[t, :] = base_g * factors

        # [핵심 수정] 매 초마다 그 당시의 구름 종류를 배열에 확실하게 박제해둡니다! (2026/04/06)
        if cloud.active:
            if cloud.event_type == 'moving':
                cloud_type_array[t] = 1
            elif cloud.event_type == 'uniform':
                cloud_type_array[t] = 2


    # 4. 열관성(Thermal Lag) 온도 계산
    T_matrix = np.zeros((total_seconds, NUM_MODULES))

    # [수정] 1. 계절별 대기 평균 기온(Base Temperature) 계산 (2026/04/06)
    month = times[0].month
    temp_air_base = 15.0 - 15.0 * np.cos(np.pi * (month - 1) / 6.0)

    # [수정] 2. 시간을 소수점(Hour) 단위로 변환 (예: 13시 30분 -> 13.5)
    hours = times.hour + (times.minute / 60.0) + (times.second / 3600.0)

    # [수정] 3. 일교차 모사 (13:00에 최고 기온, 진폭 5도 -> 총 일교차 10도 가정)
    daily_amplitude = 5.0

    # 사인파를 이용해 13시에 최대값(1), 01시에 최솟값(-1)을 가지는 곡선 생성
    temp_air_hourly = temp_air_base + daily_amplitude * np.sin(np.pi * (hours - 7.0) / 12.0)

    # 4. 계산된 시간별 기온을 Series로 변환
    temp_air_series = pd.Series(temp_air_hourly, index=times)
    wind_speed_series = pd.Series(1.0, index=times)

    for i in range(NUM_MODULES):
        poa_series = pd.Series(G_matrix[:, i], index=times)
        t_cell = pvlib.temperature.fuentes(
            poa_global=poa_series,
            temp_air=temp_air_series,       # 단일 값 대신 Series 입력
            wind_speed=wind_speed_series,   # 단일 값 대신 Series 입력
            noct_installed=NOCT_INSTALLED
        )
        T_matrix[:, i] = t_cell.values

    # 5. P&O MPPT 및 데이터 추출
    v_now = MODULE_DATA['V_oc_ref'] * NUM_MODULES * 0.8
    v_old, p_old = v_now, 0
    direction = 1

    history, lstm_X, lstm_Y = [], [], []
    prev_gmpp_v = v_now

    for t in tqdm(range(total_seconds), desc=f"{target_date} MPPT"):
        current_gs = G_matrix[t]
        current_ts = T_matrix[t]

        # 물리 엔진: P-V 곡선 계산
        i_array = np.linspace(0, MODULE_DATA['I_sc_ref'] + 1, 300)
        v_total = np.zeros_like(i_array)

        for g, temp in zip(current_gs, current_ts):
            if g < 10: g = 10
            IL, Io, Rs, Rsh, nNsVth = pvlib.pvsystem.calcparams_desoto(
                effective_irradiance=g, temp_cell=temp,
                alpha_sc=MODULE_DATA['alpha_sc'], a_ref=MODULE_DATA['a_ref'],
                I_L_ref=MODULE_DATA['I_L_ref'], I_o_ref=MODULE_DATA['I_o_ref'],
                R_sh_ref=MODULE_DATA['R_sh_ref'], R_s=MODULE_DATA['R_s'],
                EgRef=1.121, dEgdT=-0.0002677
            )
            v_mod = pvlib.pvsystem.v_from_i(
                resistance_shunt=Rsh, resistance_series=Rs, nNsVth=nNsVth,
                saturation_current=Io, photocurrent=IL, current=i_array
            )
            v_total += np.maximum(v_mod, -0.7)

        p_total = v_total * i_array
        valid = p_total > 0
        p_clean, v_clean, i_clean = p_total[valid], v_total[valid], i_array[valid]
        gmpp_idx = np.argmax(p_clean)
        true_v, true_p = v_clean[gmpp_idx], p_clean[gmpp_idx]

        # P&O 측정 및 갱신

        # 기존 코드 삭제(2026/04/02)
        # i_measured = np.interp(v_now, v_clean, i_clean)

        # [수정된 코드] 배열을 뒤집어서(오름차순으로 만들어) 보간
        # v_clean은 내림차순이므로 [::-1]을 통해 오름차순으로 역순 정렬
        idx_sort = np.argsort(v_clean) # 확실하게 오름차순 정렬 인덱스를 얻음
        v_clean_sorted = v_clean[idx_sort]
        i_clean_sorted = i_clean[idx_sort]

        i_measured = np.interp(v_now, v_clean_sorted, i_clean_sorted)
        p_measured = v_now * i_measured

        # ==========================================
        # [수정된 dP/dV 계산 로직] (2026/04/02)
        # 1. 덮어쓰기 전에 전압 변화량(dv) 먼저 계산
        dv = v_now - v_old

        # 전압 변화가 거의 없을 때(클리핑 발생 등)는 자연적인 일사량 변화에 의한
        # 전력 변화이므로 dP/dV를 0으로 처리하여 이상치 방지
        if t == 0 or abs(dv) < 1e-4:
            dp_dv = 0.0
        else:
            dp_dv = (p_measured - p_old) / dv

        # 2. P&O 알고리즘 방향 갱신
        if p_measured < p_old:
            direction *= -1

        # 3. 현재 값을 과거 변수에 저장
        v_old, p_old = v_now, p_measured

        # 4. 다음 스텝의 전압 계산
        v_now = np.clip(v_now + direction * 2.0, 10, v_clean_sorted[-1] if len(v_clean)>0 else 400)
        # ==========================================

        # [핵심] Date 정보 Feature 추가
        current_time = times[t]
        day_of_year = current_time.dayofyear # 1 ~ 365
        hour_of_day = current_time.hour + (current_time.minute / 60.0) # 5.0 ~ 19.0

        # Feature: [V, I, P, dP/dV , DayOfYear, Hour]
        history.append([v_old, i_measured, p_measured, dp_dv , day_of_year, hour_of_day])

        # (앞부분 생략: history에 데이터 append 후)

        if len(history) > WINDOW_SIZE: history.pop(0)

        # --- 1. 현재 상태 계산 --- (2026/04/06)
        ratio = p_measured / true_p

        # --- 2. 데이터 수집 (윈도우가 꽉 찼을 때만 수행) ---
        if len(history) == WINDOW_SIZE:
            # [Y=1 수집]
            if 0.85 < ratio < 0.95:
                if cloud_type_array[t] == 1:
                    if np.random.rand() < 0.1: # 10초에 1번 꼴로만 저장
                        lstm_X.append(list(history))
                        lstm_Y.append(1)

            # [Y=0 수집]
            if cloud_type_array[t] == 2: # Uniform
                if np.random.rand() < 0.02:
                    lstm_X.append(list(history))
                    lstm_Y.append(0)
            elif cloud_type_array[t] == 0: # Clear sky
                if np.random.rand() < 0.0005:
                    lstm_X.append(list(history))
                    lstm_Y.append(0)

        # --- 3. 제어 로직 (임계치 도달 시 점프) ---
        # 데이터 수집 여부와 상관없이, 시스템을 보호하기 위해 0.85 이하면 무조건 점프합니다.
        if ratio <= 0.85:
            # 점프 직전, 윈도우가 차있다면 마지막 Y=1 하나 더 저장
            if len(history) == WINDOW_SIZE and cloud_type_array[t] == 1:
                lstm_X.append(list(history))
                lstm_Y.append(1)

            v_now = true_v
            history.clear() # [중요] 점프했으므로 과거 기록 삭제

    return lstm_X, lstm_Y

# ==========================================
# [Step 4] 1월 ~ 12월 통합 실행부
# ==========================================
if __name__ == "__main__":
    # 2026년 1월 15일부터 12월 15일까지 총 12일 시뮬레이션
    months = range(1, 13)
    target_dates = [f"2026-{m:02d}-15" for m in months]
    #target_dates = [f"2026-01-15"]

    all_X = []
    all_Y = []

    for date_str in target_dates:
        day_X, day_Y = simulate_single_day(date_str)
        all_X.extend(day_X)
        all_Y.extend(day_Y)

    final_X = np.array(all_X, dtype=np.float32)
    final_Y = np.array(all_Y, dtype=np.float32)

    save_path = '/content/drive/MyDrive/mppt_dataset/mppt_thesis_dataset_1year_1.npz'
    np.savez(save_path, X=final_X, Y=final_Y)

    print("="*50)
    print("[1년치 통합 시뮬레이션 완료]")
    print(f"- 입력 시계열 데이터(X) 형태: {final_X.shape}")
    print(f"  └ Feature 구성(6개): [V, I, P, dP/dV , DayOfYear, HourOfDay]")
    print(f"- 라벨 데이터(Y) 형태: {final_Y.shape}")
    print(f"- LMPP 갇힘(1) 샘플 수: {np.sum(final_Y==1)}")
    print(f"- 정상 추종(0) 샘플 수: {np.sum(final_Y==0)}")
    print(f"- 데이터 저장 위치: {save_path}")
    print("="*50)


[2026-01-15] 시뮬레이션 시작...


2026-01-15 MPPT: 100%|██████████| 50401/50401 [02:06<00:00, 397.42it/s]



[2026-02-15] 시뮬레이션 시작...


2026-02-15 MPPT: 100%|██████████| 50401/50401 [02:09<00:00, 389.16it/s]



[2026-03-15] 시뮬레이션 시작...


...


[2026-11-15] 시뮬레이션 시작...


2026-11-15 MPPT: 100%|██████████| 50401/50401 [02:05<00:00, 400.14it/s]



[2026-12-15] 시뮬레이션 시작...


2026-12-15 MPPT: 100%|██████████| 50401/50401 [02:05<00:00, 400.09it/s]


[1년치 통합 시뮬레이션 완료]
- 입력 시계열 데이터(X) 형태: (7113, 60, 6)
  └ Feature 구성(6개): [V, I, P, dP/dV , DayOfYear, HourOfDay]
- 라벨 데이터(Y) 형태: (7113,)
- LMPP 갇힘(1) 샘플 수: 5636
- 정상 추종(0) 샘플 수: 1477
- 데이터 저장 위치: /content/drive/MyDrive/mppt_dataset/mppt_thesis_dataset_1year_1.npz
